## 1.0 Libraries and directories

In [23]:
import ee 
import geemap
import geopandas as gpd

import pprint as pp

ykf = gpd.read_file('./data/YKflats_roi_shape.shp')

ee.Authenticate()
ee.Initialize(project='ee-green-by-another-name')

## 2.0 Collection of Sentinel-2 Images on target date

In [24]:

roi_ee = ee.Geometry(ykf.iloc[0].geometry.__geo_interface__)

date = '2020-05-29'
date_plus1d = '2020-05-30' 
#Required to filter by date, but end_date is exclusive so not in image collection

s2_spec_reflec = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(roi_ee)
    .filterDate(date, date_plus1d)
)


# pp.pp(s2_spec_reflec.first().getInfo())
s2_total_imgs = s2_spec_reflec.size().getInfo()

s2_spec_reflec = s2_spec_reflec.map(lambda img: img.clip(roi_ee))
s2_spec_reflec = s2_spec_reflec.mosaic()


## 3.0 Mask the poor quality Sentinel-2 pixels

In [25]:
s2_cl_prob = (ee.ImageCollection('COPERNICUS/S2_CLOUD_PROBABILITY')
    .filterBounds(roi_ee)
    .filterDate(date, date_plus1d)
)
s2_cl_prob = s2_cl_prob.map(lambda img: img.clip(roi_ee))
s2_cl_prob = s2_cl_prob.mosaic()

s2_cl_mask = s2_cl_prob.select('probability').gt(25).rename('cl_binary')
s2_dark_mask = s2_spec_reflec.select('SCL').eq(2)
s2_cloud_shaddow_mask = s2_spec_reflec.select('SCL').eq(3)
s2_cirrus_mask = s2_spec_reflec.select('SCL').eq(10)
s2_opaque_clouds_mask = s2_spec_reflec.select('QA60').bitwiseAnd(1 << 10).eq(0)

s2_full_mask = s2_cl_mask.Or(s2_cloud_shaddow_mask).Or(s2_cirrus_mask)

## 4.0 Polygon for Sentinel-2 tile's extent

In [26]:

s2_data_mask = s2_spec_reflec.mask().reduce(ee.Reducer.anyNonZero())
s2_boundary = s2_data_mask.reduceToVectors(
    geometry=roi_ee,
    geometryType='polygon',
    scale=10,
    maxPixels=1e13
)

s2_boundary = ee.Feature(s2_boundary.toList(s2_boundary.size()).get(0))


## 5.0 Collection of Landsat Images on target date

In [27]:
def optical_rescale(img):
    "Only converts optical bands, thermal bands not included"
    img = img.select(
        ['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7']
    )
    img = img.multiply(0.0000275).add(-0.2)

    return img


ls_spec_reflec = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
                  .filterBounds(roi_ee)
                  .filterDate(date, date_plus1d)
)

# pp.pp(ls_spec_reflec.getInfo())

ls_spec_reflec = ls_spec_reflec.map(lambda img: img.clip(roi_ee))
ls_qa = ls_spec_reflec.select('QA_PIXEL').mosaic()


ls_spec_reflec = ls_spec_reflec.map(optical_rescale)

ls_total_imgs = ls_spec_reflec.size().getInfo()
ls_spec_reflec = ls_spec_reflec.mosaic()


## 6.0 Mask the poor quality Landsat pixels

In [33]:
ls_shaddow_mask = ls_qa.bitwiseAnd(1 << 4).eq(0)
ls_cloud_mask = ls_qa.bitwiseAnd(1 << 3).eq(0)
ls_dilated_cloud_mask = ls_qa.bitwiseAnd(1 << 2).eq(0)
ls_cirrus_mask = ls_qa.bitwiseAnd(1 << 2).eq(0)

ls_full_mask = ls_shaddow_mask.Or(ls_dilated_cloud_mask).Or(ls_cirrus_mask)


## 7.0 Polygon for Landsat tile's extent

In [ ]:
# Get the bounds of the Landsat image

ls_data_mask = ls_spec_reflec.mask().reduce(ee.Reducer.anyNonZero())
ls_boundary = ls_data_mask.reduceToVectors(
    geometry=roi_ee,
    geometryType='polygon',
    scale=10,
    maxPixels=1e13
)

ls_boundary = ee.Feature(ls_boundary.toList(ls_boundary.size()).get(2))

## 8.0 Visualize the results

In [37]:
s2_true_col_params = {
    'bands': ['B4', 'B3', 'B2'],
    'min': 0,
    'max': 3000,
    'gamma': 1.7
}

s2_nir_r_g = {
    'bands': ['B8', 'B4', 'B3'],
    'min': 0, 
    'max': 3000, 
    
}

ls_true_col_params = {
    'bands': ['SR_B4', 'SR_B3', 'SR_B2'],
    'min': 0,
    'max': 0.3,
    'gamma': 1.7
}

s2_cloud_prob_params = {
    'bands': ['probability'],
    'min': 0,
    'max': 100
}

binary_mask_red_params = {
    'min':0,
    'max':1,
    'palette': ['grey', 'red']
}

binary_mask_white_params = {
    'min':0,
    'max':1, 
    'palette': ['grey', 'white']
}
binary_mask_orange_params = {
    'min':0,
    'max':1, 
    'palette': ['grey', 'orange']
}

binary_mask_lblue_params = {
    'min':0,
    'max':1, 
    'palette': ['grey', '#87CEFA']
}


In [38]:
Map = geemap.Map()

#Map.addLayer(s2_spec_reflec, s2_true_col_params, f'S2 True Color Composite on {date}')
#Map.addLayer(s2_spec_reflec, s2_nir_r_g, 'S2 False Color')
Map.addLayer(ls_spec_reflec, ls_true_col_params, f'LS8 True Color Composite on {date}')

#Map.addLayer(s2_cl_prob, s2_cloud_prob_params, 'S2 Cloud Probability')
#Map.addLayer(s2_cl_mask, binary_mask_white_params, 'S2 Cloud Mask')
#Map.addLayer(s2_cloud_shaddow_mask, binary_mask_red_params, 'S2 Shaddow Mask')
#Map.addLayer(s2_dark_mask, binary_mask_orange_params, 'S2 Dark Mask')
#Map.addLayer(s2_cirrus_mask, binary_mask_lblue_params, 'S2 Cirrus Mask')
#Map.addLayer(s2_full_mask, binary_mask_red_params, 'S2 Full Mask')


#Map.addLayer(ls_full_mask, binary_mask_red_params, 'LS Full QA Mask')
#Map.addLayer(ls_cloud_mask, binary_mask_white_params, 'LS Cloud Mask')
Map.addLayer(ls_shaddow_mask, binary_mask_orange_params, 'LS Shaddow Mask')
#Map.addLayer(ls_cirrus_mask, binary_mask_lblue_params, 'LS Cirrus Mask')

# Map.addLayer(s2_boundary, {'color': 'blue'}, 'S2 Boundary')
# Map.addLayer(ls_boundary, {'color': 'green'}, 'LS Boundary')
# Map.addLayer(roi_ee, {'color': 'red'}, 'ROI Outline')
Map.centerObject(roi_ee, zoom=10)
Map


Map(center=[66.51859072213907, -146.00018537875346], controls=(WidgetControl(options=['position', 'transparent…

## 9.0 Export data masks and images

In [32]:
Export.image.toDrive({
    image: s2_full_mask,
    description: '',
    folder: 'ls_sw_coincidents_masks', 
})

SyntaxError: incomplete input (3317524684.py, line 1)